In [63]:
"""
Maximum Optimization: GL(n, F_q) Transition Matrix

Final optimizations:
1. Precompute ALL flag inverses once
2. Use numpy-style batch operations where possible
3. Algebraic characterization to avoid redundant computation
4. Memory-efficient representations
"""

import itertools
from collections import defaultdict
import time
from sage.all import *

# ============================================================
# Parameters
# ============================================================
q = 3
n = 4 # Change this - should work up to n=5 for q=2

# ============================================================
# Setup
# ============================================================
F_q = GF(q)
R = LaurentPolynomialRing(QQ, 'v')
v = R.gen()
Frac = R.fraction_field()

start_total = time.time()

print(f"="*60)
print(f"MAXIMUM OPTIMIZATION: GL({n}, F_{q})")
print(f"="*60)

# Expected sizes
print(f"\nExpected sizes:")
print(f"  |U| = q^(n(n-1)/2) = {q**(n*(n-1)//2)}")
print(f"  |W| = n! = {factorial(n)}")
num_flags = prod([(q**i - 1) // (q - 1) for i in range(1, n+1)])
print(f"  |G/B| = {num_flags}")
print(f"  Total ops for fixed flags: |U| × |G/B| = {q**(n*(n-1)//2) * num_flags}")

# ============================================================
# Utility functions  
# ============================================================

def mat_key(m):
    """Hashable key for a matrix."""
    return tuple(map(tuple, m))

def perm_to_matrix(perm):
    """Permutation (as list) to matrix."""
    m = matrix(F_q, n, n, 0)
    for i, j in enumerate(perm):
        m[i, j-1] = 1
    return m

# ============================================================
# 1. Build Weyl group W
# ============================================================

print(f"\n[1/7] Building Weyl group W...")
t0 = time.time()

# To match the original code exactly, we need to sort by the same key
# The original iterates through G and extracts permutation matrices,
# then sorts by (length, tuple(perm))
# We need to use the same tie-breaking

S_n = SymmetricGroup(n)
W_perms = sorted([Permutation(list(p.tuple())) for p in S_n],
                 key=lambda p: (len(p.inversions()), tuple(p)))
W_matrices = [perm_to_matrix(p) for p in W_perms]
W_perm_to_idx = {tuple(p): i for i, p in enumerate(W_perms)}

print(f"  |W| = {len(W_perms)} ({time.time()-t0:.2f}s)")

# ============================================================
# 2. Build unipotent group U
# ============================================================

print(f"\n[2/7] Building unipotent group U...")
t0 = time.time()

above_diag = [(i, j) for i in range(n) for j in range(i+1, n)]
num_above = len(above_diag)

U = []
for vals in itertools.product(range(q), repeat=num_above):
    m = identity_matrix(F_q, n)
    for idx, (i, j) in enumerate(above_diag):
        m[i, j] = vals[idx]
    U.append(m)

U_to_idx = {mat_key(u): i for i, u in enumerate(U)}

print(f"  |U| = {len(U)} ({time.time()-t0:.2f}s)")

# ============================================================
# 3. Build flags with cell labels + PRECOMPUTE INVERSES
# ============================================================

print(f"\n[3/7] Building flags and precomputing inverses...")
t0 = time.time()

flags = []
flag_inverses = []  # CRITICAL: precompute all inverses
flag_cell_idx = []  # Which Bruhat cell each flag is in
cell_to_flags = defaultdict(list)

for cell_idx, w in enumerate(W_perms):
    w_list = list(w)  # w_list[i] = column of pivot in row i (1-indexed)
    w_inv = w.inverse()
    length = len(w.inversions())
    
    # Correct parameterization for BwB/B:
    # The flag variety G/B can be viewed as lower-triangular matrices with 
    # a specific pivot structure. For the cell corresponding to w:
    # 
    # We use a different approach: enumerate matrices of the form w * u
    # where u ranges over a specific subset of lower-triangular unipotents.
    #
    # Alternative: Use the standard parameterization where:
    # - The matrix has 1s at positions (i, w(i)-1) for the permutation
    # - Free entries at (i, j) where i > w^{-1}(j+1)-1, i.e., below the pivot in column j
    # - But only for columns j where the pivot row is < i (captures inversions)
    #
    # Simpler approach: Free positions are (i, j) where:
    #   - j < w_list[i] - 1 (to the left of pivot in row i), AND
    #   - The entry at (w_inv(j+1)-1, j) is the pivot for column j, meaning row w_inv(j+1)-1
    #   - We only have freedom at (i, j) if i > w_inv(j+1)-1
    #
    # Even simpler: positions (i, j) with j < w_list[i]-1 AND i > w_inv[j+1]-1
    # This is: row i is below the pivot row of column j, AND column j is left of row i's pivot
    
    # Actually, the cleanest characterization: 
    # (i, j) is free iff (i, j) is an inversion of w, meaning i < j but w(i) > w(j)
    # when viewed as: matrix position (w^{-1}(i)-1, w^{-1}(j)-1) ...
    #
    # Let me use the simple direct approach: 
    # Free positions (i, j) where j < w_list[i]-1 AND w_inv(j+1) > i+1
    # This means: column j is left of row i's pivot, AND row i is below column j's pivot
    
    free_pos = []
    for i in range(n):
        pivot_col_i = w_list[i] - 1  # 0-indexed column of pivot in row i
        for j in range(pivot_col_i):  # columns to the left of pivot
            pivot_row_j = w_inv(j + 1) - 1  # 0-indexed row of pivot in column j
            if i < pivot_row_j:  # row i is ABOVE the pivot of column j
                free_pos.append((i, j))
    
    num_free = len(free_pos)
    # Verify this equals length
    if num_free != length:
        print(f"  WARNING: w={w_list}, len(free_pos)={num_free}, length={length}")
    
    for vals in itertools.product(range(q), repeat=num_free):
        m = matrix(F_q, n, n, 0)
        for i in range(n):
            m[i, w_list[i] - 1] = 1  # Set pivot
        for idx, (i, j) in enumerate(free_pos):
            m[i, j] = vals[idx]
        
        flag_idx = len(flags)
        flags.append(m)
        flag_inverses.append(m.inverse())  # Precompute!
        flag_cell_idx.append(cell_idx)
        cell_to_flags[cell_idx].append(flag_idx)

flag_to_idx = {mat_key(f): i for i, f in enumerate(flags)}

print(f"  |G/B| = {len(flags)} ({time.time()-t0:.2f}s)")

# ============================================================
# 4. Build U_w for each w
# ============================================================

print(f"\n[4/7] Building U_w subgroups...")
t0 = time.time()

Uw_indices = {}  # w_tuple -> list of indices into U

for w in W_perms:
    # Direct computation: u ∈ U_w iff w^{-1} u w ∈ U
    # We need to actually compute the conjugation, not use a shortcut formula
    
    w_mat = perm_to_matrix(w)
    w_inv_mat = w_mat.inverse()
    
    indices = []
    for u_idx, u in enumerate(U):
        conj = w_inv_mat * u * w_mat
        # Check if conj is in U (unipotent upper triangular)
        is_in_U = True
        for i in range(n):
            if conj[i, i] != 1:
                is_in_U = False
                break
            for j in range(i):
                if conj[i, j] != 0:
                    is_in_U = False
                    break
            if not is_in_U:
                break
        if is_in_U:
            indices.append(u_idx)
    
    Uw_indices[tuple(w)] = indices

print(f"  Done ({time.time()-t0:.2f}s)")

# ============================================================
# 5. Compute weights for U
# ============================================================

print(f"\n[5/7] Computing weights...")
t0 = time.time()

U_weights = []
for u in U:
    wt = sum(1 for (i,j) in above_diag if u[i,j] != 0)
    U_weights.append(wt)

print(f"  Done ({time.time()-t0:.2f}s)")

# ============================================================
# 6. MAIN COMPUTATION: Fixed flags for each u
# ============================================================

print(f"\n[6/7] Computing fixed flags (main computation)...")
t0 = time.time()

# U_fixed[u_idx] = list of flag indices that u fixes
U_fixed = [[] for _ in range(len(U))]

num_u = len(U)
num_f = len(flags)
total_ops = num_u * num_f
ops_done = 0
last_report = 0

for u_idx, u in enumerate(U):
    fixed = []
    
    for f_idx in range(num_f):
        # Check if flag f is fixed by u
        # f^{-1} u f must be upper triangular
        f_inv = flag_inverses[f_idx]  # Already computed!
        f = flags[f_idx]
        
        conj = f_inv * u * f
        
        # Fast upper-triangular check
        is_upper = True
        for i in range(1, n):
            for j in range(i):
                if conj[i, j] != 0:
                    is_upper = False
                    break
            if not is_upper:
                break
        
        if is_upper:
            fixed.append(f_idx)
    
    U_fixed[u_idx] = fixed
    
    # Progress reporting
    ops_done += num_f
    progress = ops_done / total_ops
    if progress - last_report >= 0.1:
        elapsed = time.time() - t0
        eta = elapsed / progress * (1 - progress) if progress > 0 else 0
        print(f"  {float(progress*100):.0f}% complete, ETA: {float(eta):.1f}s")
        last_report = progress

print(f"  Done ({time.time()-t0:.2f}s)")

# ============================================================
# 7. Build matrices m1, m2, m
# ============================================================

print(f"\n[7/7] Building transition matrices...")
t0 = time.time()

def normalize(lst):
    s = sum(lst)
    return lst if s == 0 else [x / s for x in lst]

# --- m1: |W| x |U| ---
print(f"  Building m1 ({len(W_perms)} x {len(U)})...")

m1_data = []
for w_idx, w in enumerate(W_perms):
    Uw_idx_set = set(Uw_indices[tuple(w)])
    row = []
    for u_idx in range(len(U)):
        if u_idx in Uw_idx_set:
            row.append(Frac((v - 1) ** U_weights[u_idx]))
        else:
            row.append(Frac(0))
    m1_data.append(normalize(row))

m1 = matrix(Frac, m1_data)

# --- m2: |U| x |W| ---
print(f"  Building m2 ({len(U)} x {len(W_perms)})...")

m2_data = []
for u_idx in range(len(U)):
    fixed_set = set(U_fixed[u_idx])
    
    sig = []
    for cell_idx in range(len(W_perms)):
        count = sum(1 for f_idx in cell_to_flags[cell_idx] if f_idx in fixed_set)
        if count == 0:
            sig.append(Frac(0))
        else:
            if count == 1:
                k = 0
            else:
                # log_q(count) - ensure integer result
                k = int(log(count, q))
            sig.append(Frac(v ** k))
    
    m2_data.append(normalize(sig))

m2 = matrix(Frac, m2_data)

# --- m = m1 * m2 ---
print(f"  Computing m = m1 * m2...")
m = m1 * m2

print(f"  Done ({time.time()-t0:.2f}s)")

# ============================================================
# Output
# ============================================================

total_time = time.time() - start_total

print(f"\n" + "="*60)
print(f"COMPLETE in {total_time:.2f}s")
print(f"="*60)

print(f"\nTransition matrix m ({len(W_perms)} x {len(W_perms)}):")
print(m)

# Verification
print(f"\n--- Verification at v=2 ---")
m_v2 = matrix(QQ, [[e.subs({v: 2}) for e in row] for row in m.rows()])
print(f"Row sums: {[float(sum(m_v2.row(i))) for i in range(m_v2.nrows())]}")

# ============================================================
# Eigenvalue analysis
# ============================================================

def show_eigenvalues(prime):
    print(f"\n--- Eigenvalues at v={prime} ---")
    m_sub = matrix(QQ, [[e.subs({v: prime}) for e in row] for row in m.rows()])
    evs = m_sub.eigenvalues()
    for ev in sorted(set(evs), key=lambda x: -abs(x.n())):
        print(f"  {ev.n(digits=6):>12}  (mult {evs.count(ev)})")

show_eigenvalues(2)


MAXIMUM OPTIMIZATION: GL(4, F_3)

Expected sizes:
  |U| = q^(n(n-1)/2) = 729
  |W| = n! = 24
  |G/B| = 2080
  Total ops for fixed flags: |U| × |G/B| = 1516320

[1/7] Building Weyl group W...
  |W| = 24 (0.01s)

[2/7] Building unipotent group U...
  |U| = 729 (0.03s)

[3/7] Building flags and precomputing inverses...
  |G/B| = 2080 (0.13s)

[4/7] Building U_w subgroups...
  Done (0.20s)

[5/7] Computing weights...
  Done (0.00s)

[6/7] Computing fixed flags (main computation)...
  10% complete, ETA: 12.0s
  20% complete, ETA: 10.6s
  30% complete, ETA: 9.2s
  40% complete, ETA: 7.9s
  50% complete, ETA: 6.5s
  60% complete, ETA: 5.2s
  70% complete, ETA: 3.9s
  80% complete, ETA: 2.6s
  90% complete, ETA: 1.3s
  Done (13.11s)

[7/7] Building transition matrices...
  Building m1 (24 x 729)...
  Building m2 (729 x 24)...
  Computing m = m1 * m2...
  Done (0.98s)

COMPLETE in 14.46s

Transition matrix m (24 x 24):
[(v^18 + 5*v^17 - 1/36*v^16 - 409/72*v^15 - 623/144*v^14 - 361/144*v^13 + 43

In [60]:
wolfmat = "{"
for row in mat2:
    wolfrow = "{"
    for entry in row:
        wolfrow += str(entry) + ", "
    wolfrow = wolfrow[:-2]
    wolfrow += "}"
    wolfmat += wolfrow + ", "
wolfmat = wolfmat[:-2]
wolfmat += "}"

In [61]:
print(wolfmat)

{{60943/299880, 11131/149940, 397/5355, 11131/149940, 8303/149940, 296/5355, 2731/74970, 296/5355, 8303/149940, 94/5355, 3641/74970, 157/5355, 8117/299880, 94/5355, 3641/74970, 83/5355, 596/37485, 146/5355, 596/37485, 83/5355, 61/5355, 26/2205, 61/5355, 1/315}, {11131/149940, 11131/74970, 296/5355, 2731/74970, 8303/74970, 94/5355, 2731/37485, 37/765, 2213/74970, 188/5355, 3641/37485, 83/5355, 8117/149940, 83/5355, 596/37485, 166/5355, 1192/37485, 61/5355, 1192/37485, 61/5355, 122/5355, 52/2205, 1/315, 2/315}, {397/5355, 296/5355, 794/5355, 296/5355, 94/5355, 592/5355, 157/5355, 592/5355, 94/5355, 188/5355, 83/5355, 314/5355, 146/5355, 188/5355, 83/5355, 166/5355, 61/5355, 292/5355, 61/5355, 166/5355, 122/5355, 1/315, 122/5355, 2/315}, {11131/149940, 2731/74970, 296/5355, 11131/74970, 2213/74970, 37/765, 2731/37485, 94/5355, 8303/74970, 83/5355, 596/37485, 83/5355, 8117/149940, 188/5355, 3641/37485, 61/5355, 1192/37485, 61/5355, 1192/37485, 166/5355, 1/315, 52/2205, 122/5355, 2/315}, {8

In [62]:
mat2 = m.subs(v=2)
valvect = mat2.eigenvectors_right()
for item in valvect:
    print("value: " + str(item[0]))
    print("vectors:")
    for v in item[1]:
        print(v)
    print()

value: 1
vectors:
(1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1)

value: 0
vectors:
(0, 1, 0, -1, 0, 1, 0, -1, 0, 0, -1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0)
(0, 0, 1, -1, 0, 0, 0, -1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0)
(0, 0, 0, 0, 1, 0, 0, 0, -1, 0, -1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0)
(0, 0, 0, 0, 0, 0, 1, 0, -1, 0, 0, 0, -1, 0, 1, 0, 0, 0, 0, 0, 0, -1/4, 0, 1/8)
(0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, -1, 0, 0, -1, 0, 0, 1, -1, 1, 0, 0)
(0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, -1, 0, 0, 0, -1, 0, 1, 0, 0, 0, 0)
(0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, -1, 0, 0, 0, -1, 1, 0, 0)
(0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, -1, 0, -1, 1, 0)

value: 0.03876086568246253?
vectors:
(0, 1, 0, -1, -0.8488407534803410?, -1.848840753480341?, 0, 1.848840753480341?, 0.8488407534803410?, 0.8736951884201001?, -0.8488407534803410?, 0, 0, -0.8736951884201001?, 0.8488407534803410?, 0.8736951884201001?, 2.230636356372423?, 0, -2.23063

In [36]:
def show_eigen_table(prime):
    print(f"\n--- Spectral Analysis at v = {prime} ---")
    
    # 1. Construct the matrix over QQbar (Algebraic Field)
    # This forces exact calculation of roots, eliminating 'noise' like 1e-9*I
    m_vals = [[e.subs({v: prime}) for e in row] for row in m.rows()]
    m_sub = matrix(QQbar, m_vals)
    
    # 2. Get eigenvectors
    # Returns list of tuples: (eigenvalue, [eigenvectors], multiplicity)
    eigen_data = m_sub.eigenvectors_right()
    
    # 3. Sort by the real part of the eigenvalue (descending)
    eigen_data.sort(key=lambda x: -x[0].real())
    
    rows = [["Eigenvalue (λ)", "Mult.", "Eigenvectors (Basis)"]]
    
    for ev, vectors, mult in eigen_data:
        # Since we use QQbar, we can explicitly ask for the real part.
        # This strips any internal complex formatting if it exists.
        ev_pretty = f"{ev.real().n(digits=6)}"
        
        # Format vectors: 
        # Convert to string, rounding entries to avoid huge algebraic strings
        vec_strs = []
        for vec in vectors:
            # We iterate through vector entries to format them nicely
            # vec is over QQbar, so we use .real().n() for clean display
            formatted_entries = [f"{x.real().n(digits=4)}" for x in vec]
            vec_strs.append(f"({', '.join(formatted_entries)})")
            
        # Join multiple basis vectors with newlines
        vec_block = "\n".join(vec_strs)
        
        rows.append([ev_pretty, mult, vec_block])
        
    t = table(rows, header_row=True, frame=True)
    show(t)

# Example usage:
show_eigen_table(2)


--- Spectral Analysis at v = 2 ---


KeyboardInterrupt: 